## 8. SAHI v2

Після train v2 перевірка, чи sliced inference ще додає на важкому bad-кадрі.
Те саме фото, що в кроці A, але ваги вже `visdrone_yolov8n_v2/best.pt`.

### SAHI v2

In [1]:
from pathlib import Path
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent

weights = ROOT / "runs/detect/runs_v2/detect_v2/visdrone_yolov8n_v2/weights/best.pt"
image_path = ROOT / "assets/predictions/bad/0000152_02214_d_0000150.jpg"
out_dir = ROOT / "assets/predictions/sahi_v2"

print("weights exist:", weights.exists())
print("image exist:", image_path.exists())

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(weights),
    confidence_threshold=0.25,
    device="cuda:0",
)

result = get_sliced_prediction(
    str(image_path),
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)

out_dir.mkdir(parents=True, exist_ok=True)
result.export_visuals(export_dir=str(out_dir))
print(len(result.object_prediction_list), "detections")
print("saved to:", out_dir / "prediction_visual.png")

weights exist: True
image exist: True
Performing prediction on 8 slices.
20 detections
saved to: C:\Users\Artem\Documents\My_project\Trainee_test_Visicom\visdrone-yolov8-detection\assets\predictions\sahi_v2\prediction_visual.png


### Порівняння на одному кадрі

**Було (bad, звичайний predict / відбір):**

![before](../../assets/predictions/bad/0000152_02214_d_0000150.jpg)

**SAHI на v1**:

![sahi_v1](../../assets/predictions/sahi_v1/prediction_visual.png)

**SAHI на v2**:

![sahi_v2](../../assets/predictions/sahi_v2/prediction_visual.png)

### Спостереження

1. detections на v2+SAHI: пошук об'єктів набагато покрищився і почали показуватись купу деталей, однак і збої такожж почав показувати, такі як: плутанина людини та таблички, плутанини цифри 0 на об'єкта  машини
2. порівняно з sahi_v1: трошки краще, більше об'єктів визначає
3. на око дрібні об'єкти: вдалечні видно ледь точки людей, або щось схоже на людей, і bbox їх позначає як людей, це гарний результат